In [1]:
import os
import re
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
from multiprocessing.pool import ThreadPool
from Bio import SeqIO

def read_fa(path):
    res = {}
    rescords = list(SeqIO.parse(path, format="fasta"))
    for x in rescords:
        id = str(x.id)
        seq = str(x.seq)
        res[id] = seq
    return res

In [2]:
lnc_dict = read_fa("/Users/andrescubillovillalobos/Documents/CompGen_Inc/RedNeuronal1/data/fasta/fasta_seq/lncRNA.fa")
mir_dict = read_fa("/Users/andrescubillovillalobos/Documents/CompGen_Inc/RedNeuronal1/data/fasta/fasta_seq/miRNA_Especifico.fa")

In [3]:
# Combinations to test

ss_loops_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Necesarios/binding_features_dic.txt", header=None, index_col=0)
index_ss_lops = ss_loops_dict.index
print(index_ss_lops)

test_lnc = [element.split('_')[0] for element in index_ss_lops]
test_mir = [element.split('_')[1] for element in index_ss_lops]


Index(['NONHSAT000179.2_hsa-miR-25-3p', 'NONHSAT000179.2_hsa-miR-299-3p',
       'NONHSAT000179.2_hsa-miR-92a-3p', 'NONHSAT000179.2_hsa-miR-92b-3p',
       'NONHSAT000201.2_hsa-miR-136-5p', 'NONHSAT000201.2_hsa-miR-148a-3p',
       'NONHSAT000201.2_hsa-miR-148b-3p', 'NONHSAT000201.2_hsa-miR-152-3p',
       'NONHSAT000201.2_hsa-miR-15a-5p', 'NONHSAT000201.2_hsa-miR-15b-5p',
       ...
       'NONHSAT064005.2_hsa-miR-128-3p', 'NONHSAT001974.2_hsa-miR-145-5p',
       'NONHSAT046854.2_hsa-miR-130b-3p', 'NONHSAT040965.2_hsa-miR-302b-3p',
       'NONHSAT004493.2_hsa-miR-339-5p', 'NONHSAT021837.2_hsa-miR-454-3p',
       'NONHSAT097840.2_hsa-miR-20a-5p', 'NONHSAT135445.2_hsa-miR-485-5p',
       'NONHSAT061665.2_hsa-miR-539-5p', 'NONHSAT033924.2_hsa-miR-21-5p'],
      dtype='object', name=0, length=45977)


In [4]:
def miRanda(i):
    v_lnc = lnc_dict[test_lnc[i]]

    f = open("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Necesarios/input_lnc.fa", "w")
    f.write(">" + test_lnc[i] + "\n")
    f.write(v_lnc + "\n")
    f.close()

    v_mir = mir_dict[test_mir[i]]

    f = open("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Necesarios/input_mir.fa", "w")
    f.write(">" + test_mir[i] + "\n")
    f.write(v_mir + "\n")
    f.close()

    cmd = f'miranda /Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Necesarios/input_mir.fa /Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Necesarios/input_lnc.fa -out "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/binding/lnc con mir especifico/binding_{test_lnc[i]}_{test_mir[i]}_especifico.txt"'
    os.system(cmd)

for i in tqdm(range(len(test_mir))):
    miRanda(i)

100%|██████████| 45977/45977 [03:24<00:00, 225.35it/s]


In [5]:
# Simplify binding outputs.

for i in range (len(test_lnc)):
    try:
        path = f'/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/binding/lnc con mir especifico/binding_{test_lnc[i]}_{test_mir[i]}_especifico.txt'
        new_file = f'/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/features/features mir especifico/feature_{test_lnc[i]}_{test_mir[i]}.txt'
        hits = 0
        with open(path, "r") as file:
            with open(new_file, "w") as new_file:
                for line in file.readlines():
                    if 'Forward:	Score:' in line and hits == 0:
                        score = line.strip().split()[1]
                        max_score = score
                        new_file.write(line + "\n")
                        hits += 1
                    if ('Query:' in line) and max_score:
                        new_file.write(line + "\n")
                    if ('Ref:' in line) and max_score:
                        new_file.write(line + "\n")
                    if ('Energy:' in line) and max_score:
                        new_file.write(line + "\n")
                        max_score = 0
                    if 'No Hits Found above Threshold' in line:
                        new_file.write(line + "\n")
    except:
        pass